# Activity 1: Implement Entropy

$$
H(Y) = -\sum_{i=1}^{n} p_i \log_2(p_i)
$$

Where:
- $H(Y)$ = Entropy of the dataset
- $p_i$ = Probability of the $i^{th}$ class
- $n$ = Number of distinct classes
- $\log_2$ = Logarithm with base 2

In [12]:

import math

def entropy(labels):
    if len(labels) == 0:
        return 0

    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1

    total = len(labels)
    result = 0

    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)

    return result

In [13]:

labels1 = ["Pass", "Pass", "Pass", "Pass"]
labels2 = ["Pass", "Pass", "Fail", "Fail"]
labels3 = ["Pass", "Fail", "Pass", "Fail"]

print("Entropy 1:", entropy(labels1))
print("Entropy 2:", entropy(labels2))
print("Entropy 3:", entropy(labels3))

Entropy 1: 0.0
Entropy 2: 1.0
Entropy 3: 1.0


# Activity 2: Implement Mutual Information



$$
I(Y;X)=H(Y)-H(Y|X)
$$

Where:

- $H(Y|X)$ = Conditional entropy of $Y$ given $X$
- $H(Y)$ = Entropy of the labels when the feature $X$ has value $x$

In [14]:

def conditional_entropy(dataset, feature, label_col="Label"):
    total = len(dataset)
    weighted_entropy = 0

    values = set(row[feature] for row in dataset)

    for value in values:
        subset = [
            row[label_col]
            for row in dataset
            if row[feature] == value
        ]

        weight = len(subset) / total
        weighted_entropy += weight * entropy(subset)

    return weighted_entropy

In [15]:

def mutual_information(dataset, feature, label_col="Label"):
    labels = [row[label_col] for row in dataset]

    h_y = entropy(labels)
    h_y_given_x = conditional_entropy(
        dataset, feature, label_col
    )

    return h_y - h_y_given_x

In [49]:

dataset = [
    {"Attendance": "High", "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low", "Assignment": "No", "Label": "Pass"},
    {"Attendance": "Low", "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "Low", "Study": "Low", "Assignment": "No", "Label": "Fail"},
    {"Attendance": "Low", "Study": "Low", "Assignment": "Yes", "Label": "Fail"},
    {"Attendance": "High", "Study": "High", "Assignment": "No", "Label": "Pass"},
    {"Attendance": "Low", "Study": "High", "Assignment": "No", "Label": "Fail"}
]

features = ["Attendance", "Study", "Assignment"]

dic = {}
for feature in features:
    dic[feature] = mutual_information(dataset, feature)

pd.DataFrame({"Feature": dic.keys(), "Mutual Information": dic.values()})

,Feature,Mutual Information
0,Attendance,0.548795
1,Study,0.048795
2,Assignment,0.048795


# Activity 3: Implement Prediction

In [50]:

class Node:
    def __init__(self, node_type, label=None, feature=None):
        self.type = node_type
        self.label = label
        self.feature = feature
        self.branches = {}

In [51]:

# Create leaf nodes
fail_leaf = Node("leaf", label="Fail")
pass_leaf = Node("leaf", label="Pass")

# Create Study node
study_node = Node("internal", feature="Study")
study_node.branches["Low"] = fail_leaf
study_node.branches["High"] = pass_leaf

# Create root node
root = Node("internal", feature="Attendance")
root.branches["Low"] = fail_leaf
root.branches["High"] = study_node

# Prediction function
def predict(tree, x):
    current = tree

    while current.type != "leaf":
        feature = current.feature
        value = x[feature]
        current = current.branches[value]

    return current.label

In [52]:

students = [
    {"Attendance": "High", "Study": "High"},
    {"Attendance": "High", "Study": "Low"},
    {"Attendance": "Low", "Study": "High"}
]

for student in students:
    print(student, "->", predict(root, student))

{'Attendance': 'High', 'Study': 'High'} -> Pass
{'Attendance': 'High', 'Study': 'Low'} -> Fail
{'Attendance': 'Low', 'Study': 'High'} -> Fail


# Activity 4: Implement Recursive Training

In [53]:

from collections import Counter

def majority_label(dataset, label_col="Label"):
    labels = [row[label_col] for row in dataset]
    return Counter(labels).most_common(1)[0][0]

In [54]:

def train(dataset, features, label_col="Label",
          default_label="Fail"):

    # Base case: empty dataset
    if len(dataset) == 0:
        return Node("leaf", label=default_label)

    labels = [row[label_col] for row in dataset]
    majority = majority_label(dataset, label_col)

    # Base case: all labels are identical
    if len(set(labels)) == 1:
        return Node("leaf", label=labels[0])

    # Base case: no features remain
    if len(features) == 0:
        return Node("leaf", label=majority)

    # Select feature with maximum MI
    best_feature = max(
        features,
        key=lambda f: mutual_information(
            dataset, f, label_col
        )
    )

    node = Node("internal", feature=best_feature)

    # Remove selected feature
    remaining_features = [
        f for f in features if f != best_feature
    ]

    # Create a child for each feature value
    values = set(row[best_feature] for row in dataset)

    for value in values:
        subset = [
            row for row in dataset
            if row[best_feature] == value
        ]

        child = train(
            subset,
            remaining_features,
            label_col,
            majority
        )

        node.branches[value] = child

    return node

In [55]:

features = ["Attendance", "Study", "Assignment"]

tree = train(dataset, features)

print("Tree trained successfully!")
print("Root feature:", tree.feature)

Tree trained successfully!
Root feature: Attendance


# Activity 5: Test the Trained Tree

In [56]:

new_students = [
    {"Attendance": "High", "Study": "Low", "Assignment": "No"},
    {"Attendance": "Low", "Study": "Low", "Assignment": "Yes"},
    {"Attendance": "Low", "Study": "High", "Assignment": "Yes"},
    {"Attendance": "Low", "Study": "High", "Assignment": "No"},
    {"Attendance": "High", "Study": "High", "Assignment": "Yes"}
]

In [57]:

for i, student in enumerate(new_students, start=1):
    prediction = predict(tree, student)

    print(
        f"Student {i}: {student} -> {prediction}"
    )

Student 1: {'Attendance': 'High', 'Study': 'Low', 'Assignment': 'No'} -> Pass
Student 2: {'Attendance': 'Low', 'Study': 'Low', 'Assignment': 'Yes'} -> Fail
Student 3: {'Attendance': 'Low', 'Study': 'High', 'Assignment': 'Yes'} -> Pass
Student 4: {'Attendance': 'Low', 'Study': 'High', 'Assignment': 'No'} -> Fail
Student 5: {'Attendance': 'High', 'Study': 'High', 'Assignment': 'Yes'} -> Pass
